# 🚦 02 — Analyse de trafic

Sources d'acquisition, devices, géographie, saisonnalité.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_sessions, load_transactions
from src.preprocessing import enrich
from src.datamarts import (
    build_traffic_by_channel, build_traffic_by_device,
    build_traffic_by_country, build_daily_traffic
)
from src.utils import set_style

set_style()
sessions = enrich(load_sessions(), load_transactions())
print(f'{len(sessions):,} sessions')

## Trafic par canal

In [ ]:
channel = build_traffic_by_channel(sessions)
channel

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax2 = ax.twinx()
channel_sorted = channel.sort_values('sessions')
ax.barh(channel_sorted['channel'], channel_sorted['sessions'],
        color='steelblue', alpha=0.6, label='Sessions')
ax2.plot(channel_sorted['conversion_rate_pct'], channel_sorted['channel'],
         color='red', marker='o', linewidth=2, label='Conversion %')
ax.set_xlabel('Sessions', color='steelblue')
ax2.set_xlabel('Conversion rate (%)', color='red')
ax.set_title('Volume vs efficacité par canal')
plt.show()

## Trafic par device

In [ ]:
build_traffic_by_device(sessions)

## Trafic par pays

In [ ]:
build_traffic_by_country(sessions).head(10)

## Saisonnalité hebdomadaire

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
by_day = sessions.groupby('session_day_of_week')['session_id'].count().reindex(day_order)
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#3498db']*5 + ['#e67e22']*2
ax.bar(by_day.index, by_day.values, color=colors, edgecolor='black')
ax.set_title('Sessions par jour de la semaine')
ax.set_ylabel('Sessions')
for i, v in enumerate(by_day.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.show()